# EXP-111 · 제출 후보 만들기 (CPU로 실행)

미리 정한 후보만 만들어 리더보드로 확인합니다. 비율을 바꿔가며 여러 번 제출하지 않습니다.

1. **Gemma 단독**
2. **다수결**: 3B, 7B, 8B, Gemma (+ Qwen3.8 결과가 있으면 포함). 동점이면 현재 최고 조합(0.96604)의 답
3. **동일 비율 평균**: 기존 앙상블(3B·7B), 8B, Gemma 각 1/3 (+ Qwen3.8 있으면 각 1/4)

In [ ]:
import os, numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
E, S = f"{SHARE_DIR}/experiment_data", f"{SHARE_DIR}/submissions"
P = ["prob_a","prob_b","prob_c","prob_d"]
FILES = {
  "3B":  ("EXP-007_train6042_resolution1280_valid_seed43.csv", "EXP-007_train6042_resolution1280_test_probabilities_seed43.csv"),
  "7B":  ("EXP-034_qwen7b_language_only_train1600_resolution1280_valid_seed43.csv", "EXP-040_qwen7b_train1600_resolution1280_test_probabilities_seed43.csv"),
  "8B":  ("EXP-102_qwen3-vl-8b-instruct_lora_r16_valid_probs.csv", "EXP-102_qwen3-vl-8b-instruct_lora_r16_test_probs.csv"),
  "G4":  ("EXP-120_gemma4-31b_zeroshot_valid_probs.csv", "EXP-120_gemma4-31b_zeroshot_test_probs.csv"),
  "Q38": ("EXP-121_qwen3.8-27b_zeroshot_valid_probs.csv", "EXP-121_qwen3.8-27b_zeroshot_test_probs.csv"),
  "G4b": ("EXP-122_gemma4-31b_t280_short_valid_probs.csv", "EXP-122_gemma4-31b_t280_short_test_probs.csv"),
  "Q35": ("EXP-123_qwen3.5-27b_qlora_valid_probs.csv", "EXP-123_qwen3.5-27b_qlora_test_probs.csv"),
  "G4c": ("EXP-124_gemma4-31b_t1120_valid_probs.csv", "EXP-124_gemma4-31b_t1120_test_probs.csv"),
  "8B2": ("EXP-104_qwen3-vl-8b-instruct_lora_r16_seed17_res1280_valid_probs.csv", "EXP-104_qwen3-vl-8b-instruct_lora_r16_seed17_res1280_test_probs.csv"),
  "IVL": ("EXP-105_internvl3-8b_lora_r16_valid_probs.csv", "EXP-105_internvl3-8b_lora_r16_test_probs.csv"),
  "4B":  ("EXP-102_qwen3-vl-4b-instruct_lora_r16_valid_probs.csv", "EXP-102_qwen3-vl-4b-instruct_lora_r16_test_probs.csv"),
}
valid = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv"); v_ids = valid.id.values
ans = valid.answer.map(dict(zip("abcd",range(4)))).values
t_ids = pd.read_csv(f"{E}/{FILES['3B'][1]}").id.values
V, T = {}, {}
for k, (vf, tf) in FILES.items():
    if os.path.exists(f"{E}/{vf}") and os.path.exists(f"{E}/{tf}") and len(pd.read_csv(f"{E}/{tf}")) == 6714:
        V[k] = pd.read_csv(f"{E}/{vf}").set_index("id").loc[v_ids, P].values.astype(float)
        T[k] = pd.read_csv(f"{E}/{tf}").set_index("id").loc[t_ids, P].values.astype(float)
    else: print("[건너뜀]", k)
print("사용:", list(V)); assert "G4" in V, "Gemma test가 아직 끝나지 않았습니다"
for k in V: print(f"  {k} 단독 valid {(V[k].argmax(1)==ans).sum()}/672")
REF = pd.read_csv(f"{S}/EXP-102_qwen3-vl-8b-instruct_lora_r16_ens_w5.csv")   # 0.96604 (바뀐 문항 수 비교 기준)

In [ ]:
best = lambda D: 0.5*(0.52*D["3B"] + 0.48*D["7B"]) + 0.5*D["8B"]
def vote(D):
    voters = [k for k in ["3B","7B","8B","G4","Q38","Q35"] if k in D]
    preds = np.stack([D[k].argmax(1) for k in voters], 1); tie = best(D).argmax(1); out = []
    for i, row in enumerate(preds):
        c = np.bincount(row, minlength=4); top = np.where(c == c.max())[0]
        out.append(top[0] if len(top) == 1 else tie[i])
    return np.array(out), voters
def equal(D):
    D = {k: v for k, v in D.items() if k not in ("8B2","IVL","4B")}   # 추가 모델은 아래 후보에서만 명시적으로 사용
    parts = [0.52*D["3B"] + 0.48*D["7B"], D["8B"], D["G4"]] + [D[k] for k in ["Q38","Q35"] if k in D]
    return sum(parts) / len(parts)

suffix = ("_q38" if "Q38" in V else "") + ("_q35" if "Q35" in V else "")
cands = {}
if "G4b" in V:   # EXP-122(280·짧은 프롬프트) 결과가 있으면: 단독 + 동일 평균에서 Gemma를 교체한 버전
    D2 = lambda D: {**D, "G4": D["G4b"]}
    cands["EXP-111_gemma280_single"] = (V["G4b"].argmax(1), T["G4b"].argmax(1))
    cands["EXP-111_equal_mean_gemma280"] = (equal({k:v for k,v in D2(V).items() if k not in ("Q38","Q35")}).argmax(1), equal({k:v for k,v in D2(T).items() if k not in ("Q38","Q35")}).argmax(1))
    # Gemma 두 설정(560·OCR 프롬프트 + 280·짧은 프롬프트)을 먼저 평균해 Gemma 한 자리로 사용
    D3 = lambda D: {k: v for k, v in {**D, "G4": (D["G4"] + D["G4b"]) / 2}.items() if k not in ("Q38","Q35")}
    cands["EXP-111_equal_mean_gemma2x"] = (equal(D3(V)).argmax(1), equal(D3(T)).argmax(1))   # LB 0.96782
    # gemma2x 방식을 다른 모델에도: 같은 계열 두 모델을 먼저 평균해 한 자리로
    base3 = lambda D: [0.52*D["3B"] + 0.48*D["7B"], D["8B"], (D["G4"] + D["G4b"]) / 2]
    if "8B2" in V:   # 8B 자리 = 8B·8B2(시드·해상도 다름) 평균
        f = lambda D: (base3(D)[0] + (D["8B"] + D["8B2"]) / 2 + base3(D)[2]) / 3
        cands["EXP-111_gemma2x_8b2x"] = (f(V).argmax(1), f(T).argmax(1))
    if "4B" in V:    # 8B 자리 = 8B·4B 평균 (4B는 8B보다 약하므로 2:1)
        f = lambda D: (base3(D)[0] + (2*D["8B"] + D["4B"]) / 3 + base3(D)[2]) / 3
        cands["EXP-111_gemma2x_8b4b"] = (f(V).argmax(1), f(T).argmax(1))
    if "IVL" in V:   # InternVL을 네 번째 자리로 (계열이 달라 다양성)
        f = lambda D: (sum(base3(D)) + D["IVL"]) / 4
        cands["EXP-111_gemma2x_ivl"] = (f(V).argmax(1), f(T).argmax(1))
if "Q35" in V:   # 27B 파인튜닝: 단독 + (기존·8B·Gemma·Q35 각 1/4) — Qwen3.8 제외 버전
    noq38 = lambda D: {k: v for k, v in D.items() if k != "Q38"}
    cands["EXP-111_q35_single"] = (V["Q35"].argmax(1), T["Q35"].argmax(1))
    cands["EXP-111_equal_mean_q35"] = (equal(noq38(V)).argmax(1), equal(noq38(T)).argmax(1))
cands.update({
  f"EXP-111_gemma_single":        (V["G4"].argmax(1),  T["G4"].argmax(1)),
  f"EXP-111_vote{suffix}":         (vote(V)[0],        vote(T)[0]),
  f"EXP-111_equal_mean{suffix}":   (equal(V).argmax(1), equal(T).argmax(1)),
})
# ===== Gemma 여러 설정 평균(gx)을 기본으로, Qwen3.8/3.5를 한 자리씩 추가 =====
gx = lambda D: sum(D[k] for k in ["G4","G4b","G4c"] if k in D) / len([k for k in ["G4","G4b","G4c"] if k in D])
gtag = "gemma" + str(len([k for k in ["G4","G4b","G4c"] if k in V])) + "x"
def gx_mean(D, extra):
    parts = [0.52*D["3B"] + 0.48*D["7B"], D["8B"], gx(D)] + [D[k] for k in extra]
    return sum(parts) / len(parts)
for extra in [[], ["Q38"], ["Q35"], ["Q38","Q35"]]:
    if all(k in V for k in extra) and (extra or "G4c" in V):
        name = f"EXP-111_{gtag}" + "".join("_" + k.lower() for k in extra)
        cands[name] = (gx_mean(V, extra).argmax(1), gx_mean(T, extra).argmax(1))
print("투표 참여:", vote(V)[1], "| 기준(0.96604 조합) valid:", (best(V).argmax(1)==ans).sum(), "/672\n")
for name, (vp, tp) in cands.items():
    sub = pd.DataFrame({"id": t_ids, "answer": ["abcd"[i] for i in tp]})
    assert (sub.id.values == REF.id.values).all()
    sub.to_csv(f"{S}/{name}.csv", index=False)
    print(f"{name:28s} valid {(vp==ans).sum()}/672 | 0.96604 대비 바뀐 test {(sub.answer.values != REF.answer.values).sum():4d}개 | 저장")